# Quick check: can the arm reach the RIGHT imaging rack?

The right imaging rack stands behind the sample racks, to the right of the sample-0 rack. Its slots run
front to back, and a rod placed in it lies left to right with the **metal end on the left** and the
**coated end on the right** (towards the camera). The gripper holds the electrode near its metal end,
on the left of the rack. Slots are counted from the rack's front screw (the one away from the wall):
slot 0 is right next to it, and slots 1, 2, 3, ... go towards the wall.

- **Cell 1** moves the **empty** gripper from where it is now (pointing down) to 10 cm above the
  metal end of an electrode in the chosen slot. On the way, high above the bench, the gripper turns a
  quarter turn (the other way round than for the left imaging rack).
- **Cell 2** lets you fine-tune the spot by typing commands (for example `f 5` = 5 mm towards you).
  `down` opens the gripper and lowers it to 5 mm above the grab height, `d 1` / `u 1` go 1 mm lower
  or higher, `grab` closes the gripper on the electrode and `open` lets go again.
  `done` prints the final spot for Claude.
- **Cell 3** brings the arm back to where it started, turning back.

Cell 2 starts from the spot found last time (slot 2: x = 294, y = 488.1), so you can fine-tune it again.

Before running:
- Put one electrode in the slot the way the arm should leave it: it is the target for the fingers.
- Swing the camera and its boom out of the way.
- Gripper empty, the red wire on the gripper has some slack, hand on the e-stop.
- Do not run anything in another notebook that is connected to the arm at the same time.

Run the cells in order, without restarting the kernel in between.
To try another slot, change `SLOT` at the top of cell 1.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='imaging_quick_check_right', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Quick check 1/3 (RIGHT imaging rack): EMPTY gripper from where it is now to 10 cm above the slot =====
# The right imaging rack: its slots run front to back, and a rod in it lies left to right with the
# metal end on the LEFT and the coated end on the RIGHT (towards the camera).
# Before running: camera and boom swung out of the way, gripper empty, hand on the e-stop.
# Put one electrode in the slot the way the arm should leave it: it is the target for the fingers.
# Route: straight up from where the arm is now
#        -> across, high above the bench, turning the gripper a quarter turn on the way
#           (the other way round than for the left imaging rack)
#        -> stops so you can look at the gripper and its red wire
#        -> down to 10 cm above the slot (only after your "yes").
# These moves are not checked against the code's safety zone, so they are slow and need your "yes".

SLOT = 2          # counted from the rack's FRONT screw (the one away from the wall): 0 = the slot right
                  # next to it; 1, 2, 3, ... go towards the wall, 20 mm apart.
                  # 2 = the slot your electrode lies in, in your photo.
SAFE_Z = 500.0    # travel height, well above the flask

# Found with cell 2 last time: the fingers held the electrode in slot 2 at x = 294, y = 488.1.
# Cell 2 starts from there, so you can fine-tune it again. Other slots are 20 mm apart.
FOUND_SLOT, FOUND_SPOT = 2, [294.0, 488.1, 190.6]     # x, y, z where the fingers hold the electrode
RACK_RPY = [180.0, 0.0, 90.0]        # gripper pointing down, rod front-to-back (the normal racks)
IMAGING_RPY = [180.0, 0.0, 180.0]    # gripper pointing down, rod left-to-right, coated end on the RIGHT

x, y, z = FOUND_SPOT[0], FOUND_SPOT[1] + 20.0 * (SLOT - FOUND_SLOT), FOUND_SPOT[2]


def at_speed(mm_s):
    """Normal motion settings, with the speed (mm/s) changed."""
    return {**arm.config.pos_settings_dict['default'], 'speed': mm_s}


start = arm.get_cartesian_pos()          # remembered, so cell 3 can bring the arm back here
print('Arm is now at:', [round(v, 1) for v in start])


def turned_from(rpy):
    """Largest angle (degrees) between the arm's current orientation and rpy."""
    return max(abs(((a - b) + 180) % 360 - 180) for a, b in zip(start[3:], rpy))


# The arm may start pointing down either the normal way or already turned (left there by an earlier run).
start_rpy = RACK_RPY if turned_from(RACK_RPY) <= 5 else IMAGING_RPY if turned_from(IMAGING_RPY) <= 5 else None
if start_rpy is None or start[2] < 240:
    raise SystemExit('The gripper must point straight down, turned the normal way (as above the racks), and be '
                     'at least 5 cm above the racks (z >= 240) before this check. The arm did not move.')

steps = [
    ('straight up from here',                  [start[0], start[1], SAFE_Z, *start_rpy], 50),
    ('across, turning a quarter turn',         [x, y, SAFE_Z, *IMAGING_RPY], 30),
]
print(f'Plan (slot {SLOT}):')
for name, pose, speed in steps + [('then, after your yes: down to 10 cm above the slot',
                                   [x, y, z + 100, *IMAGING_RPY], 20)]:
    print(f'  {name:52s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
print('Check the red wire on the gripper now: it needs some slack, because the gripper will turn a quarter turn.')
if input('Camera moved away, gripper empty, red wire has slack, hand on the e-stop? Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

arm.assert_ready()
for name, pose, speed in steps:
    print('->', name)
    arm.move_to_pos(pose, **at_speed(speed))
print('Now high above the imaging rack at:', [round(v, 1) for v in arm.get_cartesian_pos()])
print('Look at the gripper: it should have turned only a quarter turn, the fingers now one in front and')
print('one behind (towards the wall), and the red wire should still be loose, not pulled tight.')
if input('All good? Type yes to go down to 10 cm above the slot: ').strip().lower() == 'yes':
    arm.move_to_pos([x, y, z + 100, *IMAGING_RPY], **at_speed(20))
    print('Stopped at:', [round(v, 1) for v in arm.get_cartesian_pos()])
    print(f'Look from the front and from the side: are the fingers right above the metal end of the electrode in slot {SLOT}?')
    print('Next: run cell 2 to fine-tune the spot.')
else:
    print('Stayed high in the air. Run cell 3 to bring the arm back.')

In [ ]:
# ===== Quick check 2/3: fine-tune the spot and the height, and try a grab =====
# Directions as seen standing in front of the bench, facing the wall. Type one of these and press Enter:
#   f 5 / b 5 / l 5 / r 5  -> 5 mm towards you / towards the wall / left / right
#   down   -> open the gripper, then go down slowly to 5 mm above the grab height (asks first)
#   d 1    -> 1 mm lower      u 1 -> 1 mm higher      (after down; at most 3 mm per step)
#   grab   -> close the gripper on the electrode (after down)
#   open   -> open the gripper again
#   up     -> back up to 10 cm above (gripper must be open)
#   done   -> finish (gripper must be open), and print the spot to send to Claude
# Sideways steps are at most 10 mm (3 mm when down), never more than 30 mm from the planned spot,
# and never while the gripper is closed. The height stays between 5 mm below and 20 mm above z.

MOVES = {'f': (0, -1), 'b': (0, 1), 'l': (-1, 0), 'r': (1, 0)}
HIGH = z + 100          # 10 cm above the grab height
LOWEST, HIGHEST = z - 5, z + 20


def offset_text(ox, oy):
    """How far the gripper is from the planned spot, in words."""
    parts = []
    if abs(oy) >= 0.05:
        parts.append(f'{abs(oy):.1f} mm towards ' + ('the wall' if oy > 0 else 'you'))
    if abs(ox) >= 0.05:
        parts.append(f'{abs(ox):.1f} mm to the ' + ('right' if ox > 0 else 'left'))
    return (', '.join(parts) + ' of the planned spot') if parts else 'on the planned spot'


def height_text(h):
    """Height of the gripper compared with the usual grab height, in words."""
    if abs(h - z) < 0.05:
        return 'at the usual grab height'
    return f'{abs(h - z):.1f} mm ' + ('above' if h > z else 'below') + ' the usual grab height'


here = arm.get_cartesian_pos()
if max(abs(here[0] - x), abs(here[1] - y), abs(here[2] - HIGH)) > 5:
    print('The arm is not 10 cm above the slot (cell 1 stopped early), so it stays where it is.')
    print('Run cell 3 to bring it back.')
else:
    tx, ty, tz = x, y, HIGH
    closed = False
    while True:
        cmd = input('f/b/l/r or d/u + mm, or down, grab, open, up, done: ').strip().lower().split()
        if not cmd:
            continue
        low = tz != HIGH
        if cmd == ['done']:
            if closed:
                print('The gripper is closed. Type open first, so the electrode stays in the rack.')
                continue
            break
        if cmd == ['down']:
            if low:
                print('Already down. Use d 1 to go 1 mm lower.')
            elif input('Fingers right above the metal end of the electrode, nothing in the way? '
                       'Type yes to open the gripper and go down slowly: ').strip().lower() == 'yes':
                arm.open_gripper()
                tz = z + 5
                arm.move_to_pos([tx, ty, tz, *IMAGING_RPY], **at_speed(8))
                print('Down,', height_text(tz) + '. The open fingers should be one in front of and one behind')
                print('the electrode, near its metal end, touching nothing. Use d 1 to go lower.')
            continue
        if cmd == ['up']:
            if closed:
                print('The gripper is closed. Type open first.')
                continue
            tz = HIGH
            arm.move_to_pos([tx, ty, tz, *IMAGING_RPY], **at_speed(20))
            print('Back up, 10 cm above the slot.')
            continue
        if cmd in (['grab'], ['open']):
            if cmd == ['grab'] and not low:
                print('Go down first.')
                continue
            if cmd == ['grab']:
                arm.close_gripper()
                closed = True
                print(f'Gripper closed, opening now {arm.gripper_opening:.1f} mm. Does it hold the metal part well?')
                print('Type open to let go again.')
            else:
                arm.open_gripper()
                closed = False
                print('Gripper open.')
            continue
        if len(cmd) != 2 or cmd[0] not in ('f', 'b', 'l', 'r', 'd', 'u'):
            print('Did not understand. Examples: f 5, b 2, l 3, r 1, d 1, u 1, down, grab, open, up, done.')
            continue
        try:
            mm = float(cmd[1])
        except ValueError:
            print('The second part must be a number of mm, e.g. d 1.')
            continue
        if cmd[0] in ('d', 'u'):
            if not low:
                print('Type down first; d and u only work near the electrode.')
                continue
            if not 0 < mm <= 3:
                print('Use a distance between 0 and 3 mm.')
                continue
            nz = tz - mm if cmd[0] == 'd' else tz + mm
            if not LOWEST <= nz <= HIGHEST:
                print('That is outside the allowed height (5 mm below to 20 mm above the usual grab height), '
                      'so the arm did not move. Tell Claude.')
                continue
            tz = nz
            arm.move_to_pos([tx, ty, tz, *IMAGING_RPY], **at_speed(3))
            print('Moved. Now', height_text(tz) + '.')
            continue
        if closed:
            print('The gripper is closed: type open before moving sideways.')
            continue
        step_max = 3.0 if low else 10.0
        if not 0 < mm <= step_max:
            print(f'Use a distance between 0 and {step_max:g} mm.')
            continue
        dx, dy = MOVES[cmd[0]]
        nx, ny = tx + dx * mm, ty + dy * mm
        if max(abs(nx - x), abs(ny - y)) > 30:
            print('That would be more than 30 mm from the planned spot, so the arm did not move. Tell Claude.')
            continue
        tx, ty = nx, ny
        arm.move_to_pos([tx, ty, tz, *IMAGING_RPY], **at_speed(5))
        print('Moved. Now', offset_text(tx - x, ty - y) + '.')
    print(f'Spot: x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}   <- send this line to Claude')
    print('Then run cell 3 to bring the arm back.')

In [ ]:
# ===== Quick check 3/3: back to where the arm started =====
# Route: straight up (no turning) -> across, high above the bench, turning back if needed
#        -> down to the start position.
cur = arm.get_cartesian_pos()
print('Going back: straight up, across (turning the gripper back), down to the start position.')
arm.move_to_pos([cur[0], cur[1], SAFE_Z, *cur[3:]], **at_speed(20))
arm.move_to_pos([start[0], start[1], SAFE_Z, *start_rpy], **at_speed(30))
arm.move_to_pos([start[0], start[1], start[2], *start_rpy], **at_speed(20))
print('Back at:', [round(v, 1) for v in arm.get_cartesian_pos()])